# SIM Asset Intelligence: synthetic cost anomaly experiment

This Google Colab notebook creates entirely synthetic SIM cost records, compares a transparent ratio rule with Isolation Forest, and exports review flags. Do not upload employee, phone, ICCID, operator billing or other organizational data without written authorization. Flags are suggestions for human review only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
rng = np.random.default_rng(SEED)
n = 500
operators = rng.choice(['Operator A', 'Operator B', 'Operator C'], size=n)
plans = rng.choice(['Voice', 'Data', 'Voice + Data'], size=n, p=[0.25, 0.35, 0.40])
expected_by_plan = {'Voice': 900.0, 'Data': 1400.0, 'Voice + Data': 2300.0}
expected_cost = np.array([expected_by_plan[p] for p in plans])
monthly_cost = np.maximum(100, rng.normal(expected_cost, expected_cost * 0.12))
known_anomaly = np.zeros(n, dtype=int)
anomaly_idx = rng.choice(n, size=25, replace=False)
known_anomaly[anomaly_idx] = 1
monthly_cost[anomaly_idx[:17]] *= rng.uniform(2.5, 4.0, size=17)
monthly_cost[anomaly_idx[17:]] *= rng.uniform(0.08, 0.25, size=8)

data = pd.DataFrame({'synthetic_sim_id': [f'SIM-DEMO-{i+1:04d}' for i in range(n)], 'operator': operators, 'plan_type': plans, 'expected_cost': expected_cost, 'monthly_cost': np.round(monthly_cost, 2), 'known_synthetic_anomaly': known_anomaly})
data['cost_ratio'] = data['monthly_cost'] / data['expected_cost']
data.head()

In [ ]:
# Transparent baseline: flag costs far outside the expected plan cost.
data['baseline_flag'] = ((data['cost_ratio'] > 1.8) | (data['cost_ratio'] < 0.35)).astype(int)
print('Baseline results against the injected synthetic labels:')
print(classification_report(data['known_synthetic_anomaly'], data['baseline_flag'], zero_division=0))
print('Confusion matrix [normal, anomaly]:')
print(confusion_matrix(data['known_synthetic_anomaly'], data['baseline_flag']))

In [ ]:
# Train/test split. Isolation Forest is a candidate detector, not a decision maker.
features = data[['monthly_cost', 'expected_cost', 'cost_ratio']]
X_train, X_test, y_train, y_test = train_test_split(features, data['known_synthetic_anomaly'], test_size=0.25, random_state=SEED, stratify=data['known_synthetic_anomaly'])
model = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED)
model.fit(X_train)
data.loc[X_test.index, 'model_flag'] = (model.predict(X_test) == -1).astype(int)
print('Isolation Forest results on the held-out synthetic split:')
print(classification_report(y_test, data.loc[X_test.index, 'model_flag'], zero_division=0))
print('Confusion matrix [normal, anomaly]:')
print(confusion_matrix(y_test, data.loc[X_test.index, 'model_flag']))

In [ ]:
# Visual inspection and export. No operational action is performed.
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(data['expected_cost'], data['monthly_cost'], c=data['known_synthetic_anomaly'], alpha=0.7)
ax.set_xlabel('Expected plan cost (synthetic KES)')
ax.set_ylabel('Monthly cost (synthetic KES)')
ax.set_title('Synthetic SIM costs: normal records and injected anomalies')
plt.show()

output_path = '/content/sim_anomaly_review_flags.csv'
data.to_csv(output_path, index=False)
print(f'Exported synthetic review flags to {output_path}')
print('Review each flag manually. Never auto-suspend SIMs or penalize employees based on model output.')